In [1]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install -U transformers datasets fsspec
!pip install pandas scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
ERROR: pip's dependen

In [7]:
# ============================================================
# CELL 2: Imports
# ============================================================
import os
import glob
import shutil
import zipfile
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from transformers.modeling_outputs import ModelOutput
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, mean_squared_error, mean_absolute_error, classification_report


In [4]:

# ============================================================
# CELL 3: Upload training data
# ============================================================
from google.colab import files

uploaded = files.upload()
zip_path = next(iter(uploaded))  # name of the uploaded zip file

extract_dir = "/content/training_data"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Extracted training data to {extract_dir}")


Saving dataset.zip to dataset.zip
Extracted training data to /content/training_data


In [8]:
# ============================================================
# CELL 4: View how data is spread across events
# ============================================================
csv_files_preview = glob.glob(os.path.join(extract_dir, "**", "*.csv"), recursive=True)
if not csv_files_preview:
    raise FileNotFoundError(
        f"No .csv files found under {extract_dir}. Contents: {os.listdir(extract_dir)}"
    )

df_preview = pd.concat([pd.read_csv(f) for f in csv_files_preview], ignore_index=True)

event_col = "event_class" if "event_class" in df_preview.columns else "event"
print(df_preview[event_col].value_counts())
print(f"Total rows: {len(df_preview)}")

event_class
normal_play    2151
scrum           149
goal_kick       121
lineout         111
tmo_replay       62
penalty          54
kick_off         52
maul             44
try              40
Name: count, dtype: int64
Total rows: 2784


In [9]:

# ============================================================
# CELL 5: Build label maps
#
# normal_play is pinned to id 0 - CELL 11 defaults any blank or
# unrecognized event_class value to label 0, so 0 must stay
# normal_play regardless of where it sits in EVENT_CLASSES.
# ============================================================
EVENT_CLASSES = [
    "try",
    "goal_kick",
    "card_event",
    "scrum",
    "maul",
    "lineout",
    "kick_off",
    "tmo_replay",
    "penalty",
]

label2id = {"normal_play": 0}
id2label = {0: "normal_play"}
for i, evt in enumerate(EVENT_CLASSES, start=1):
    label2id[evt] = i
    id2label[i] = evt

num_labels = len(label2id)
print("Labels:", label2id)



Labels: {'normal_play': 0, 'try': 1, 'goal_kick': 2, 'card_event': 3, 'scrum': 4, 'maul': 5, 'lineout': 6, 'kick_off': 7, 'tmo_replay': 8, 'penalty': 9}


In [10]:

# ============================================================
# CELL 6: Model definition - DualHeadRoBERTa
#   - event_classifier: predicts the event type (try, penalty, ...)
#   - highlight_scorer: predicts a 0-1 "how exciting is this" score
# ============================================================
MODEL_NAME = "roberta-base"  # change if training a different base model


@dataclass
class DualHeadRoBERTaOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    highlight_scores: torch.FloatTensor = None


class DualHeadRoBERTa(nn.Module):
    def __init__(self, model_name, num_labels, id2label, label2id):
        super().__init__()
        self.config = AutoModel.from_pretrained(model_name).config
        self.config.num_labels = num_labels
        self.config.id2label = id2label
        self.config.label2id = label2id

        self.roberta = AutoModel.from_pretrained(model_name)

        self.event_classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.highlight_scorer = nn.Linear(self.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask, labels=None, highlight_labels=None, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = getattr(outputs, "pooler_output", None)
        if pooled_output is None:
            pooled_output = outputs.last_hidden_state[:, 0, :]

        event_logits = self.event_classifier(pooled_output)
        highlight_score = torch.sigmoid(self.highlight_scorer(pooled_output))

        return DualHeadRoBERTaOutput(logits=event_logits, highlight_scores=highlight_score)



In [11]:

# ============================================================
# CELL 7: Loss functions
#
# FocalLoss: for the event classifier - handles class imbalance
#   automatically, so normal_play should NOT be down-weighted
#   separately (that destroys precision, see class_weights below).
# WeightedMSELoss: for the highlight scorer - upweights samples
#   with a meaningfully high highlight score.
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss


class WeightedMSELoss(nn.Module):
    def __init__(self, base_weight=1.0, high_score_weight=5.0, threshold=0.01):
        super().__init__()
        self.base_weight = base_weight
        self.high_score_weight = high_score_weight
        self.threshold = threshold

    def forward(self, inputs, targets):
        mse = F.mse_loss(inputs, targets, reduction="none")
        weights = torch.where(targets > self.threshold, self.high_score_weight, self.base_weight)
        return (mse * weights).mean()


In [12]:

# ============================================================
# CELL 8: Custom Trainer
#
# stage=1 trains the event classifier head, stage=2 trains the
# highlight scorer head (see CELL 13/15).
# ============================================================
class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, stage=1, **kwargs):
        super().__init__(*args, **kwargs)
        self.stage = stage
        self.loss_fct_events = FocalLoss(alpha=class_weights, gamma=2.0)
        self.loss_fct_highlights = WeightedMSELoss(base_weight=1.0, high_score_weight=5.0, threshold=0.01)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        highlight_labels = inputs.pop("highlight_labels")

        outputs = model(**inputs)
        logits = outputs.logits
        highlight_scores = outputs.highlight_scores

        if self.stage == 1:
            if self.loss_fct_events.alpha is not None and self.loss_fct_events.alpha.device != logits.device:
                self.loss_fct_events.alpha = self.loss_fct_events.alpha.to(logits.device)
            loss = self.loss_fct_events(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        else:
            loss = self.loss_fct_highlights(highlight_scores.view(-1), highlight_labels.view(-1).to(highlight_scores.dtype))

        return (loss, outputs) if return_outputs else loss



In [13]:

# ============================================================
# CELL 9: Evaluation metrics
#
# The model has two label columns (labels, highlight_labels), so
# Trainer passes both heads' predictions/targets here once per
# evaluation pass (each epoch, since eval_strategy="epoch"). This
# reports classification metrics (accuracy, precision, recall, f1)
# for the event head AND regression metrics (mse, mae) for the
# highlight head, in every epoch's validation row - even though
# only one head is actively training per stage.
# ============================================================
def compute_metrics(eval_pred):
    (event_logits, highlight_preds) = eval_pred.predictions
    (event_labels, highlight_labels) = eval_pred.label_ids

    event_preds = event_logits.argmax(axis=-1)
    accuracy = accuracy_score(event_labels, event_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        event_labels, event_preds, average="weighted", zero_division=0
    )

    highlight_preds = highlight_preds.reshape(-1)
    highlight_labels = highlight_labels.reshape(-1)
    mse = mean_squared_error(highlight_labels, highlight_preds)
    mae = mean_absolute_error(highlight_labels, highlight_preds)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mse": mse,
        "mae": mae,
    }



In [14]:

# ============================================================
# CELL 10: Load the tokenizer + model (fresh start every run)
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = DualHeadRoBERTa(MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)



config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
# ============================================================
# CELL 11: Load CSVs and compute class weights
# ============================================================
csv_files = glob.glob(os.path.join(extract_dir, "**", "*.csv"), recursive=True)
dfs = []
for f in csv_files:
    d = pd.read_csv(f)
    if "event_class" in d.columns:
        d["label"] = d["event_class"].apply(lambda x: label2id.get(str(x).strip(), 0) if pd.notna(x) else 0)
    elif "event" in d.columns:
        d["label"] = d["event"].apply(lambda x: label2id.get(str(x).strip(), 0) if pd.notna(x) else 0)

    if "highlight_score" in d.columns:
        d["highlight_label"] = pd.to_numeric(d["highlight_score"], errors="coerce").fillna(0.0)
    else:
        d["highlight_label"] = 0.0

    dfs.append(d)

df = pd.concat(dfs, ignore_index=True).dropna(subset=["text"])
df_balanced = df.sample(frac=1, random_state=42).reset_index(drop=True)

total_samples = len(df_balanced)
class_counts = df_balanced["label"].value_counts().to_dict()
weights = []
for i in range(num_labels):
    count = class_counts.get(i, 0)
    w = total_samples / (num_labels * count) if count > 0 else 1.0
    weights.append(w)

class_weights = torch.tensor(weights, dtype=torch.float)
print("Class weights:", class_weights)


Class weights: tensor([0.1294, 6.9600, 2.3008, 1.0000, 1.8685, 6.3273, 2.5081, 5.3538, 4.4903,
        5.1556])


In [16]:

# ============================================================
# CELL 12: Tokenize and split into train/test
# ============================================================
dataset = Dataset.from_pandas(df_balanced)


def tokenize_func(examples):
    tokenized = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    tokenized["labels"] = examples["label"]
    tokenized["highlight_labels"] = examples["highlight_label"]
    return tokenized


tokenized_datasets = dataset.map(tokenize_func, batched=True)
split_datasets = tokenized_datasets.train_test_split(test_size=0.2, seed=42)

OUTPUT_DIR = "/content/model_output"


def get_training_args(output_dir_suffix, epochs=5):
    return TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/{output_dir_suffix}",
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        logging_strategy="steps",
        logging_steps=5,
        save_strategy="epoch",
        load_best_model_at_end=True,
        learning_rate=2e-5,
        warmup_ratio=0.1,
    )



Map:   0%|          | 0/2784 [00:00<?, ? examples/s]

In [17]:

# ============================================================
# CELL 13: Stage 1 - Train the event classifier
# ============================================================
trainer_stage1 = CustomTrainer(
    model=model,
    args=get_training_args("stage1", epochs=5),
    train_dataset=split_datasets["train"],
    eval_dataset=split_datasets["test"],
    class_weights=class_weights,
    stage=1,
    compute_metrics=compute_metrics,
)
trainer_stage1.train()



[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mse,Mae
1,1.171351,1.944874,0.037702,0.001421,0.037702,0.002740,0.109473,0.270675
2,2.104339,1.746062,0.070018,0.028865,0.070018,0.028929,0.111542,0.272591
3,1.477965,1.329910,0.120287,0.042157,0.120287,0.048321,0.112806,0.275212
4,0.190520,1.196665,0.132855,0.038774,0.132855,0.056575,0.111194,0.272955
5,0.396056,1.107261,0.143627,0.036278,0.143627,0.056628,0.112570,0.275360


TrainOutput(global_step=700, training_loss=1.133953823532377, metrics={'train_runtime': 408.689, 'train_samples_per_second': 27.246, 'train_steps_per_second': 1.713, 'total_flos': 0.0, 'train_loss': 1.133953823532377, 'epoch': 5.0})

In [19]:
# ============================================================
# CELL 14: Per-class classification report (event head)
#
# compute_metrics only reports weighted-average precision/recall/f1
# each epoch. This runs one prediction pass over the held-out test
# set and prints sklearn's full per-class breakdown, like:
#   precision  recall  f1-score  support   (one row per event)
# ============================================================
pred_output = trainer_stage1.predict(split_datasets["test"])
event_logits, _ = pred_output.predictions
event_labels, _ = pred_output.label_ids
event_preds = event_logits.argmax(axis=-1)

target_names = [id2label[i] for i in range(num_labels)]
print(classification_report(
    event_labels, event_preds, labels=list(range(num_labels)), target_names=target_names, zero_division=0
))

              precision    recall  f1-score   support

 normal_play       0.00      0.00      0.00       423
         try       0.09      0.78      0.17         9
   goal_kick       0.16      0.71      0.27        21
  card_event       0.00      0.00      0.00         0
       scrum       0.15      0.69      0.25        35
        maul       0.22      0.42      0.29        12
     lineout       0.10      0.40      0.16        20
    kick_off       0.14      0.29      0.19        14
  tmo_replay       0.21      0.77      0.33        13
     penalty       0.13      0.70      0.22        10

    accuracy                           0.14       557
   macro avg       0.12      0.47      0.19       557
weighted avg       0.04      0.14      0.06       557



In [20]:

# ============================================================
# CELL 15: Stage 2 - Train the highlight scorer
# (backbone + event head are frozen so stage 1 knowledge is kept)
# ============================================================
for name, param in model.named_parameters():
    if "highlight_scorer" not in name:
        param.requires_grad = False

trainer_stage2 = CustomTrainer(
    model=model,
    args=get_training_args("stage2", epochs=5),
    train_dataset=split_datasets["train"],
    eval_dataset=split_datasets["test"],
    class_weights=class_weights,
    stage=2,
    compute_metrics=compute_metrics,
)
trainer_stage2.train()

for param in model.parameters():
    param.requires_grad = True



[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mse,Mae
1,0.169145,0.186315,0.143627,0.036278,0.143627,0.056628,0.102300,0.257699
2,0.136162,0.177636,0.143627,0.036278,0.143627,0.056628,0.094925,0.246812
3,0.237736,0.174963,0.143627,0.036278,0.143627,0.056628,0.092073,0.243767
4,0.124858,0.174081,0.143627,0.036278,0.143627,0.056628,0.091067,0.242902
5,0.158113,0.173858,0.143627,0.036278,0.143627,0.056628,0.090781,0.242749


In [ ]:

# ============================================================
# CELL 16: Save the best model and zip it
# ============================================================
best_dir = os.path.join(OUTPUT_DIR, "best")
os.makedirs(best_dir, exist_ok=True)
torch.save(model.state_dict(), os.path.join(best_dir, "pytorch_model.bin"))
model.config.save_pretrained(best_dir)
tokenizer.save_pretrained(best_dir)

out_zip_base = "/content/trained_model"
shutil.make_archive(out_zip_base, "zip", best_dir)
print(f"Model zipped to {out_zip_base}.zip")


In [ ]:

# ============================================================
# CELL 17: Download the trained model to your local machine
# ============================================================
files.download(f"{out_zip_base}.zip")
